In [1]:
import torch
import torch.nn as nn
import shap
import json
import random
import time
import datetime
import logging
logger = logging.getLogger(__name__)

from PIL import Image
from transformers import AutoModel, AutoTokenizer
from typing import Dict, List, Optional
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, TensorDataset, DataLoader

/opt/conda/envs/minicpmv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModel.from_pretrained('../output/minicpmv_sensitive_fiofp', trust_remote_code=True, torch_dtype=torch.bfloat16)
model = model.to(device='cuda:3', dtype=torch.bfloat16)
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained('../output/minicpmv_sensitive_fiofp', trust_remote_code=True)
model.eval()

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.70it/s]


MiniCPMV(
  (llm): MiniCPMForCausalLM(
    (model): MiniCPMModel(
      (embed_tokens): Embedding(122753, 2304)
      (layers): ModuleList(
        (0-39): 40 x MiniCPMDecoderLayer(
          (self_attn): MiniCPMSdpaAttention(
            (q_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (k_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (v_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (o_proj): Linear(in_features=2304, out_features=2304, bias=False)
            (rotary_emb): MiniCPMRotaryEmbedding()
          )
          (mlp): MiniCPMMLP(
            (gate_proj): Linear(in_features=2304, out_features=5760, bias=False)
            (up_proj): Linear(in_features=2304, out_features=5760, bias=False)
            (down_proj): Linear(in_features=5760, out_features=2304, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): MiniCPMRMSNorm()
          (post_attention_layernorm): Min

In [5]:
class DisturbanceDataset(Dataset):
    def __init__(self, raw_data):
        super(DisturbanceDataset, self).__init__()
        self.raw_data = raw_data  # json文件

    def __len__(self):
        return len(self.raw_data)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        try:
            image = Image.open(self.raw_data[i]["image"]).convert("RGB").resize((1024, 1280))
            disturbance_image = Image.open(self.raw_data[i]["disturbance_image"]).convert("RGB").resize((1024, 1280))

            conversations = self.raw_data[i]["conversations"]
            conversations[0]['content'] = conversations[0]['content'].replace('<image>\\n', '')
            msgs = [item for item in conversations if item["role"] == "user"]
            images = [image for _ in range(len(msgs))]
            disturbance_images = [disturbance_image for _ in range(len(msgs))]
            tokenized_keywords = self.raw_data[i]["tokenized_keywords"]
            ret = dict(
                images=images,
                disturbance_images=disturbance_images,
                msgs=msgs,
                tokenized_keywords=tokenized_keywords,
            )
        except:
            logger.error(f"data fetch error")
            return self.__getitem__(random.randint(0, len(self)))
        return ret

In [6]:
def disturbance_collator(examples):
    images = [img for example in examples for img in example["images"]]
    disturbance_images = [img for example in examples for img in example["disturbance_images"]]
    msgs = [msg for example in examples for msg in example["msgs"]]
    tokenized_keywords = [keyword for example in examples for keyword in example["tokenized_keywords"]]
    return {
        "images": images,
        "disturbance_images": disturbance_images,
        "msgs": msgs,
        "tokenized_keywords": tokenized_keywords,
    }

In [7]:

data_path = '../data/vqa/FIOFP/split_output/proportional/FIOFP_100_disturbance_locate.json'
with open(data_path, 'r') as f:
    data = json.load(f)
dataset = DisturbanceDataset(data)
dataloader = DataLoader(dataset, batch_size=4, collate_fn=disturbance_collator)

In [8]:
target_layers = [25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]

In [9]:
def custom_generate(
        model,
        data_list=None,
        img_list=None,
        tokenizer=None,
        max_inp_length=None,
        vision_hidden_states=None,
        **kwargs
):
    
    assert data_list is not None
    bs = len(data_list)
    if img_list == None:
        img_list = [[] for i in range(bs)]
    assert bs == len(img_list)

    model_inputs = model._process_list(tokenizer, data_list, max_inp_length)
    if vision_hidden_states is None:
        pixel_values = []
        for i in range(bs):
            img_inps = []
            for img in img_list[i]:
                img_inps.append(model.transform(img).to(model.device))
            if img_inps:
                pixel_values.append(img_inps)
            else:
                pixel_values.append([])
        model_inputs["pixel_values"] = pixel_values
    else:
        model_inputs["vision_hidden_states"] = vision_hidden_states

    with torch.inference_mode():
        (
            model_inputs["inputs_embeds"],
            _,
        ) = model.get_vllm_embedding(model_inputs)

        result = model.llm.generate(inputs_embeds=model_inputs["inputs_embeds"], pad_token_id=0,
                                        eos_token_id=tokenizer.eos_token_id, **kwargs)

    return result

In [10]:
def custom_chat(
        model,
        images,
        msgs,
        tokenizer,
        vision_hidden_states=None,
        max_new_tokens=256,
        sampling=True,
        max_inp_length=2048,
        **kwargs
):
    if isinstance(msgs, str):
        msgs = json.loads(msgs)
            # msgs to prompt
    prompts = []
    input_images = []
    for i, msg in enumerate(msgs):
        prompt = ""
        role = msg["role"]
        content = msg["content"]
        assert role == "user", "The role of first msg should be user"
        if model.config.slice_mode:
            imgs, final_placeholder = model.get_slice_image_placeholder(
                    images[i], tokenizer
            )
            content = final_placeholder + "\n" + content
        else:
            imgs = [images[i]]
            content = (
                        tokenizer.im_start
                        + tokenizer.unk_token * model.config.query_num
                        + tokenizer.im_end
                        + "\n"
                        + content
            )
        input_images.append(imgs)
        prompt += "<用户>" if role == "user" else "<AI>"
        prompt += content
        prompt += "<AI>"
        prompts.append(prompt)

    if sampling:
        generation_config = {
                "top_p": 0.8,
                "top_k": 100,
                "temperature": 0.7,
                "do_sample": True,
                "repetition_penalty": 1.05
        }
    else:
        generation_config = {
                "num_beams": 3,
                "repetition_penalty": 1.2,
        }

    generation_config.update(
            (k, kwargs[k]) for k in generation_config.keys() & kwargs.keys()
    )

    with torch.inference_mode():
        res = custom_generate(
                model=model,
                data_list=prompts,
                max_inp_length=max_inp_length,
                img_list=input_images,
                tokenizer=tokenizer,
                max_new_tokens=max_new_tokens,
                vision_hidden_states=vision_hidden_states,
                **generation_config
        )

    return res

In [11]:
def compute_unlearning_outputs(model, images, msgs, tokenized_keywords, target_layers, tokenizer, device):
    """
        :param model:
        :param images:
        :param msgs:
        :param tokenized_keywords:
        :param target_layers:
        :param device:
        :return:
    """
    ffn_outputs = []

    def capture_ffn_output(module, input, output):
        ffn_outputs.append(output[:, -1, :].cpu())

    def register_ffn_hook(model, target_layers):
        hooks = []
        for idx, layer in enumerate(model.llm.model.layers):
            if idx in target_layers:
                hook = layer.mlp.up_proj.register_forward_hook(capture_ffn_output)
                hooks.append(hook)
        return hooks

    hooks = register_ffn_hook(model, target_layers)
    res = custom_chat(
            model=model,
            images=images,
            msgs=msgs,
            tokenizer=tokenizer,
            sampling=True,
            temperature=0.7,
    )

    tokenized_keywords = torch.tensor(tokenized_keywords, device=res.device)
    mask = torch.isin(res, tokenized_keywords)

    unlearning_first_token_outputs = []
    target_layers_len = len(target_layers)
    layer_outputs = [[] for _ in range(target_layers_len)]
    for i in range(len(ffn_outputs)):
        layer_index = (i - target_layers_len) % target_layers_len
        if i < target_layers_len:
            layer_outputs[layer_index].append(ffn_outputs[i])
            unlearning_first_token_outputs.append(ffn_outputs[i])
        else:
            layer_outputs[layer_index].append(ffn_outputs[i])
    layer_outputs = [torch.stack(layer) for layer in layer_outputs]

    unlearning_sensitive_outputs = []
    mask_t = mask.T.cpu()
    keywords_indices = mask_t.nonzero(as_tuple=True)
    for layer_index in range(target_layers_len):
        layer_output = layer_outputs[layer_index]
        selected_outputs = layer_output[keywords_indices]
        unlearning_sensitive_outputs.append(selected_outputs.view(-1, selected_outputs.size(-1)))

    for hook in hooks:
        hook.remove()
    return unlearning_first_token_outputs, unlearning_sensitive_outputs

In [12]:
def compute_disturbance_outputs(model, images, msgs, target_layers, tokenizer, device):
    """
        :param model:
        :param images:
        :param msgs:
        :param target_layers:
        :param tokenizer:
        :param device:
        :return:
    """
    ffn_outputs = []

    def capture_ffn_output(module, input, output):
        ffn_outputs.append(output[:, -1, :].cpu())

    def register_ffn_hook(model, target_layers):
        hooks = []
        for idx, layer in enumerate(model.llm.model.layers):
            if idx in target_layers:
                hook = layer.mlp.up_proj.register_forward_hook(capture_ffn_output)
                hooks.append(hook)
        return hooks

    hooks = register_ffn_hook(model, target_layers)
    _ = custom_chat(
            model=model,
            images=images,
            msgs=msgs,
            tokenizer=tokenizer,
            sampling=True,
            temperature=0.7,
            max_new_tokens=1,
    )

    disturbance_first_token_outputs = []
    for i in range(len(target_layers)):
        disturbance_first_token_outputs.append(ffn_outputs[i])

    for hook in hooks:
        hook.remove()
    return disturbance_first_token_outputs

In [13]:
def parameters_input_solver(unlearning_outputs, disturbance_outputs, target_layers, device):
    """
        :param unlearning_outputs:
        :param disturbance_outputs:
        :param target_layers:
        :return:
    """
    layer_indices = []

    unlearning_outputs_tensor = torch.stack(unlearning_outputs, dim=0)
    disturbance_outputs_tensor = torch.stack(disturbance_outputs, dim=0)

    scores = torch.mean(torch.abs(unlearning_outputs_tensor - disturbance_outputs_tensor), dim=1).to(device)
    for idx, layer_idx in enumerate(target_layers):
        if layer_idx < 25:
            layer_indices.append(torch.where(scores[idx] >= 0.2)[0].tolist())
        else:
            layer_indices.append(torch.topk(scores[idx], 2500).indices.tolist())
    # del scores
    return layer_indices

In [14]:
def train_auxiliary_model(model, criterion, optimizer, dataloader, device, num_epochs=5):
    for epoch in range(num_epochs):
        total_loss = 0.0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device).to(torch.float32), labels.to(device).long()

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)
        epoch_loss = total_loss / len(dataloader.dataset)
        print(f'Epoch {epoch}/{num_epochs}, Loss: {epoch_loss}')

In [15]:
def evaluate_auxiliary_model(model, dataloader, device):
    model.eval()
    corrects, total = 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device).to(torch.float32), labels.to(device).long()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels).item()
            total += inputs.size(0)
    accuracy = corrects / total
    print(f'Accuracy: {accuracy}')
    return accuracy

In [16]:
def compute_indices_by_shapley_values(shap_values, layer, device):
    shap_values_tensor = torch.stack([torch.tensor(shap_val, device=device) for shap_val in shap_values], dim=0)[:, :, 0].mean(dim=0)
    if layer < 25:
        return torch.where(shap_values_tensor >= 5e-4)[0].tolist()
    else:
        return torch.topk(shap_values_tensor, 2500).indices.tolist()

In [17]:
class AuxiliaryNet(nn.Module):
    def __init__(self):
        super(AuxiliaryNet, self).__init__()
        self.fc1 = nn.Linear(5760, 2304)
        self.fc2 = nn.Linear(2304, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.fc2(self.relu(out))
        return out


def weight_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

In [18]:
def parameters_shap_solver(unlearning_outputs, disturbance_outputs, target_layers, device):
    """
        :param unlearning_outputs:
        :param disturbance_outputs:
        :param target_layers:
        :param device:
        :return:
    """
    layer_indices = []
    for idx, unlearning_output in enumerate(unlearning_outputs):
        start_time = time.time()
        disturbance_output = disturbance_outputs[idx]
        dec_data = torch.cat([unlearning_output, disturbance_output], dim=0)
        dec_labels = torch.cat([torch.ones(unlearning_output.shape[0]), torch.zeros(disturbance_output.shape[0])], dim=0)
        dec_train_data, dec_test_data, dec_train_labels, dec_test_labels = train_test_split(dec_data, dec_labels, test_size=0.2, random_state=42)
        dec_train_loader = DataLoader(TensorDataset(dec_train_data, dec_train_labels), batch_size=512, shuffle=True)
        dec_test_loader = DataLoader(TensorDataset(dec_test_data, dec_test_labels), batch_size=512, shuffle=False)
        auxiliary_model = AuxiliaryNet().to(device)
        auxiliary_model.apply(weight_init)
        optimizer = torch.optim.Adam(auxiliary_model.parameters(), lr=1e-5)
        criterion = nn.CrossEntropyLoss()
        train_auxiliary_model(auxiliary_model, criterion, optimizer, dec_train_loader, device, num_epochs=5)
        accuracy = evaluate_auxiliary_model(auxiliary_model, dec_test_loader, device)
        print(f'Layer {idx}, Accuracy: {accuracy}')
        print(dec_train_data.shape[0])
        print('Training time {}'.format(str(datetime.timedelta(seconds=int(time.time() - start_time)))))
        start_time = time.time()
        y1 = 500
        indices = torch.randint(0, dec_train_data.shape[0], (y1,))
        sub_dec_train_data = dec_train_data[indices].to(torch.float32).to(device)
        explainer = shap.DeepExplainer(auxiliary_model, sub_dec_train_data)
        print(unlearning_output.shape[0])  # x
        y2 = 500
        indices = torch.randint(0, unlearning_output.shape[0], (y2,))
        sub_unlearning_output = unlearning_output[indices].to(torch.float32).to(device)
        print(sub_unlearning_output.shape)
        shap_values = explainer.shap_values(sub_unlearning_output)
        shap_time = time.time() - start_time
        print('Shap time {}'.format(str(datetime.timedelta(seconds=int(shap_time)))))
        layer_indices.append(compute_indices_by_shapley_values(shap_values, target_layers[idx], device))


    return layer_indices

In [19]:
def compute_ffn_outputs(model, disturbance_batch, target_layers, tokenizer, device):
    """
        :param model:
        :param disturbance_batch:
        :param target_layers:
        :param device:
        :return:
    """
    images = disturbance_batch['images']
    disturbance_images = disturbance_batch['disturbance_images']
    msgs = disturbance_batch['msgs']
    tokenized_keywords = disturbance_batch['tokenized_keywords']
    unlearning_first_token_outputs, unlearning_sensitive_outputs = compute_unlearning_outputs(
            model, images, msgs, tokenized_keywords, target_layers, tokenizer, device
    )
    disturbance_first_token_outputs = compute_disturbance_outputs(
            model, disturbance_images, msgs, target_layers, tokenizer, device
    )
    return unlearning_first_token_outputs, unlearning_sensitive_outputs, disturbance_first_token_outputs

In [20]:
def merge_indices(input_indices, shap_indices):
    """
        :param input_indices:
        :param shap_indices:
        :return:
    """
    result = []
    for i in range(len(input_indices)):
        result.append(list(set(input_indices[i]).intersection(set(shap_indices[i]))))
    return result

In [21]:
def get_parameter_indices(model, dataloader, target_layers, tokenizer, device):

    model.eval()
    target_layers_len = len(target_layers)

    unlearning_first_token_outputs = [[] for _ in range(target_layers_len)]
    unlearning_sensitive_outputs = [[] for _ in range(target_layers_len)]
    disturbance_outputs = [[] for _ in range(target_layers_len)]
    for idx, batch in enumerate(dataloader):
        start_time = time.time()
        unlearning_first_token_output, unlearning_sensitive_output, disturbance_output = compute_ffn_outputs(
            model, batch, target_layers, tokenizer, device)
        for i in range(target_layers_len):
            unlearning_first_token_outputs[i].append(unlearning_first_token_output[i])
            unlearning_sensitive_outputs[i].append(unlearning_sensitive_output[i])
            disturbance_outputs[i].append(disturbance_output[i])
        compute_time = time.time() - start_time
        print('compute_time time {}'.format(str(datetime.timedelta(seconds=int(compute_time)))))
        print(f"-------batch {idx} completed!-------")

    for i in range(target_layers_len):
        unlearning_first_token_outputs[i] = torch.cat(unlearning_first_token_outputs[i], dim=0)
        unlearning_sensitive_outputs[i] = torch.cat(unlearning_sensitive_outputs[i], dim=0)
        disturbance_outputs[i] = torch.cat(disturbance_outputs[i], dim=0)
        
    torch.cuda.empty_cache()
    
    input_indices = parameters_input_solver(unlearning_first_token_outputs, disturbance_outputs, target_layers, device)
    shap_indices = parameters_shap_solver(unlearning_sensitive_outputs, disturbance_outputs, target_layers, device)
    parameter_indices = merge_indices(input_indices, shap_indices)
    
    return parameter_indices

In [ ]:
parameter_indices = get_parameter_indices(model, dataloader, target_layers, tokenizer, device)


In [23]:
with open("./crucial_parameter_indices_fiofp_proportional.json", "w") as f:
    json.dump(parameter_indices, f, indent=4)